# Multimodal MIMIC-4 Task Tutorial
**Contributors:** William Pang, Joshua Chen, Rian Atri

This notebook demonstrates how to use the `multimodal_mimic4` task.

**Related Resources** (*Note: We likely need to update file paths once merged to Sunlab Pyhealth Main*)
- [Multimodal MIMIC4 Example](https://github.com/Multimodal-PyHealth/PyHealth/blob/main/examples/mortality_prediction/multimodal_mimic4.py)
- [Multimodal MIMIC4 Task](https://github.com/Multimodal-PyHealth/PyHealth/blob/main/pyhealth/tasks/multimodal_mimic4.py)

## 0. Environment Setup and Loading Packages

In [ ]:
from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import tempfile
import shutil 
import random
import numpy as np
import torch

*Pyhealth Specific Packages*

In [ ]:
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ClinicalNotesICDLabsMIMIC4

*Set Randomness Seed*

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Notebook-Specific Utility Functions

In [ ]:
def print_patient_admission_info(sample):
    patient = dataset.get_patient(sample['patient_id'])
    admissions = patient.get_events(
        event_type="admissions",
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\nnum_admissions:  {len(admissions)}")
    for adm in admissions:
        print(f"  hadm_id:       {adm.hadm_id}")
        print(f"  admittime:     {adm.timestamp}")
        print(f"  dischtime:     {adm.dischtime}")

def print_patient_note_info(sample, note_type="discharge", char_limit=80):
    patient = dataset.get_patient(sample['patient_id'])
    notes = patient.get_events(
        event_type=note_type,
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\n{note_type} notes: {len(notes)}")
    for note in notes:
        print(f"  note_id:   {note.note_id}")
        print(f"  hadm_id:   {note.hadm_id}")
        print(f"  charttime: {note.timestamp}")
        print(f"  storetime: {note.storetime}")
        print(f"  text:      {note.text[:char_limit]}..(Limited to {char_limit} Characters).")

## 2. Load Demo Dataset

We use the MIMIC-IV demo data in `test-resources/core/mimic4demo`. 

This includes:
- Synthetic Notes (`discharge`, `radiology`)

In [ ]:
PYHEALTH_REPO_ROOT = '/Users/wpang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "test-resources/core/mimic4demo")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "test-resources/core/mimic4demo")
CACHE_DIR = tempfile.mkdtemp()

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    note_root=NOTE_ROOT,
    ehr_tables=["diagnoses_icd", "procedures_icd", "labevents"],
    note_tables=["discharge", "radiology"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)

## 3. Time Filtering

### 3.1 Full Patient Window
By default the task uses all data between the first admission time and the last discharge time across the patient's processed admissions.

In [ ]:
%%capture
task = ClinicalNotesICDLabsMIMIC4()
samples = dataset.set_task(task, num_workers=1)

In [ ]:
random_patient_record = np.random.randint(0, len(samples)-1)
sample = samples[random_patient_record]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")

# Admissions
print_patient_admission_info(sample)

# Discharge Notes
print_patient_note_info(sample, note_type="discharge")

# Radiology Notes
print_patient_note_info(sample, note_type="radiology")

### 3.2 Random Patient Window

In [ ]:
%%capture
task = ClinicalNotesICDLabsMIMIC4(window_hours=12)
samples = dataset.set_task(task, num_workers=1)
sample = samples[random_patient_record]
sample_patient_id = sample['patient_id']

In [ ]:
print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")
print(f"Window start:    {sample['window_start']}")
print(f"Window end:      {sample['window_end']}")

# Admissions
print_patient_admission_info(sample)

# Discharge Notes
print_patient_note_info(sample, note_type="discharge")

# Radiology Notes
print_patient_note_info(sample, note_type="radiology")

## 4. Cleanup

- Remove `CACHE_DIR`

In [ ]:
shutil.rmtree(CACHE_DIR) 